# CodeGuard on Google Colab (T4)

Runs the GPU work of the project on a free Colab T4 (16 GB, Linux), top to bottom with **Runtime → Run all**:

1. mount Google Drive and unpack `MyDrive/CodeGuard/codeguard_colab_bundle.zip` (made by `scripts/make_colab_bundle.py`);
2. install the pinned requirements, print `nvidia-smi` and package versions;
3. run the tests;
4. `scripts/preflight.py` with the timing run (1.3B model), output saved to `logs/preflight_colab.log`;
5. `SMOKE=1 GPUS="0" bash scripts/run_server.sh` — the Linux rehearsal required by WP5;
6. the insurance runs `zs_1.3b_small` and `lora_1.3b_r16_small` (small tier, 1.3B);
7. if time remains: `zs_6.7b_small_4bit` (6.7B zero-shot, 4-bit, small tier);
8. `make_report.py` on the finished runs.

**Nothing is lost on a disconnect.** After every stage, `runs/`, `logs/`, `results/` (and the smoke
directories) are copied to `MyDrive/CodeGuard/out/` without `checkpoints/`. At the start they are copied
back, and every stage is skipped if its `metrics.json` (or, for tests / preflight / smoke, its done-marker)
is already there. After a disconnect: reconnect, switch the T4 on again, **Run all** again.

Not saved on a disconnect: model weights (downloaded again, a few minutes) and training checkpoints
(a LoRA run that was cut mid-training starts training again from the beginning).

Numbers printed here are results only if they come from a run's `metrics.json`; smoke runs (64 pairs,
2 minutes) are code checks, not results.

In [ ]:
# ---------------------------------------------------------------- settings (edit if needed)
DRIVE_DIR = "/content/drive/MyDrive/CodeGuard"  # the bundle zip lives here; outputs go to DRIVE_DIR/out
BUNDLE = "codeguard_colab_bundle.zip"
WORK = "/content/CodeGuard"                     # local (fast) working copy

RUN_TESTS = True
RUN_PREFLIGHT = True
RUN_SMOKE = True
RUN_INSURANCE = True          # zs_1.3b_small and lora_1.3b_r16_small
RUN_6_7B_4BIT = True          # zs_6.7b_small_4bit, only after everything above

TRAIN_BATCH, TRAIN_ACCUM = 8, 4   # 8 x 4 = 32 (the design's effective batch); lower to 4 x 8 on OOM
EVAL_BATCH = 16                   # generation / scoring batch; lower to 8 on OOM
EVAL_BATCH_6_7B = 8               # the 6.7B KV cache is larger; 16 would be close to the T4 limit
TIME_BUDGET_MIN = 180             # training minutes for lora_1.3b_r16_small (then it saves and evaluates)

DRY_RUN = False                   # True: print the commands only (used to check this notebook off Colab)

M_13 = "deepseek-ai/deepseek-coder-1.3b-instruct"; REV_13 = "e063262dac8366fc1f28a4da0ff3c50ea66259ca"
M_67 = "deepseek-ai/deepseek-coder-6.7b-instruct"; REV_67 = "e5d64addd26a6a1db0f9b863abf6ee3141936807"

In [ ]:
# ---------------------------------------------------------------- helpers
import json
import os
import shutil
import subprocess
import time
import zipfile
from pathlib import Path

OUT = Path(DRIVE_DIR) / "out"
SYNC_DIRS = ["runs", "logs", "results", "runs_smoke", "logs_smoke", "results_smoke"]
STAGES = []  # (stage, status) for the summary at the end
ENV = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8", "TOKENIZERS_PARALLELISM": "false",
       "HF_HUB_DISABLE_SYMLINKS_WARNING": "1"}


def sh(cmd, log=None):
    """Run a shell command in WORK, stream its output (last tqdm frame per line), append it to `log`."""
    print("$", cmd, flush=True)
    if DRY_RUN:
        return 0
    stream = open(Path(WORK) / log, "a", encoding="utf-8") if log else None
    proc = subprocess.Popen(cmd, shell=True, cwd=WORK, env=ENV, executable="/bin/bash", text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1, errors="replace")
    for line in proc.stdout:
        print(line.split("\r")[-1], end="", flush=True)
        if stream:
            stream.write(line)
    code = proc.wait()
    if stream:
        stream.close()
    print(f"[exit code {code}]", flush=True)
    return code


def sync_out():
    """Copy runs/, logs/, results/ (+ smoke dirs) to Drive, without checkpoints."""
    for name in SYNC_DIRS:
        source = Path(WORK) / name
        if source.exists():
            shutil.copytree(source, OUT / name, dirs_exist_ok=True,
                            ignore=shutil.ignore_patterns("checkpoints", "__pycache__"))
    print(f"synced to {OUT}", flush=True)


def has_metrics(run, runs_dir="runs"):
    return (Path(WORK) / runs_dir / run / "metrics.json").exists()


def marker(stage):
    return Path(WORK) / "logs" / f"colab_{stage}.done.json"


def stage_done(stage):
    return marker(stage).exists() and json.loads(marker(stage).read_text())["exit_code"] == 0


def finish(stage, exit_code, started):
    """Record a stage, copy outputs to Drive."""
    if not DRY_RUN:
        marker(stage).parent.mkdir(parents=True, exist_ok=True)
        marker(stage).write_text(json.dumps({"exit_code": exit_code, "minutes": (time.time() - started) / 60,
                                             "finished": time.strftime("%Y-%m-%d %H:%M:%S")}))
    STAGES.append((stage, "ok" if exit_code == 0 else f"FAILED (exit {exit_code})"))
    sync_out()


def skip(stage, why):
    print(f"skip {stage}: {why}")
    STAGES.append((stage, f"skipped ({why})"))

## 1. Google Drive, bundle, earlier outputs

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("not on Colab: using DRIVE_DIR as a plain directory")

bundle = Path(DRIVE_DIR) / BUNDLE
if not bundle.exists():
    zips = sorted(Path(DRIVE_DIR).glob("*.zip"), key=lambda p: p.stat().st_mtime)
    assert zips, f"Upload {BUNDLE} to {DRIVE_DIR} first (see the README / Russian guide)."
    bundle = zips[-1]
print("bundle:", bundle, f"{bundle.stat().st_size / 2**20:.1f} MiB")

shutil.rmtree(WORK, ignore_errors=True)       # always start from the bundle's code
with zipfile.ZipFile(bundle) as archive:
    archive.extractall(WORK)
for name in ("src/train_repair.py", "scripts/run_server.sh", "data/processed/small/train.parquet",
             "data/processed/full/test.parquet"):
    assert (Path(WORK) / name).exists(), f"{name} is missing from the bundle"

OUT.mkdir(parents=True, exist_ok=True)
for name in SYNC_DIRS:                          # earlier outputs: lets finished stages be skipped
    if (OUT / name).exists():
        shutil.copytree(OUT / name, Path(WORK) / name, dirs_exist_ok=True)
os.chdir(WORK)
print("finished runs found on Drive:", sorted(p.parent.name for p in Path("runs").glob("*/metrics.json")))

## 2. Install pinned requirements, show the GPU and versions

Colab's own torch is replaced by the version the project was tested with. All project code runs in
separate `python` processes, so the notebook kernel itself does not need a restart.

In [ ]:
installed_flag = Path("/content/.codeguard_installed")
if installed_flag.exists() or DRY_RUN:
    print("requirements already installed in this session" if not DRY_RUN else "(dry run)")
else:
    skip_names = ("jupyter", "nbconvert", "ipykernel")  # never touch Colab's own notebook packages
    lines = []
    for name in ("requirements.txt", "requirements-neural.txt"):
        for line in (Path(WORK) / name).read_text().splitlines():
            line = line.strip()
            if line and not line.startswith(("#", "-r")) and not line.startswith(skip_names):
                lines.append(line)
    Path("/content/requirements-colab.txt").write_text("\n".join(lines) + "\n")
    torch_pin = next(l for l in lines if l.startswith("torch=="))
    assert sh(f"pip install -q {torch_pin} --index-url https://download.pytorch.org/whl/cu126") == 0
    assert sh("pip install -q -r /content/requirements-colab.txt") == 0
    installed_flag.touch()

sh("nvidia-smi")
sh("python --version && python -c \"import sys; assert sys.version_info >= (3, 11), 'pandas 3 needs Python 3.11+'\"")
sh("python -c \"import torch, transformers, peft, accelerate, bitsandbytes, sacrebleu, pandas; "
   "print('torch', torch.__version__, 'cuda', torch.version.cuda, 'available', torch.cuda.is_available()); "
   "print('gpu', torch.cuda.get_device_name(0), 'native bf16', torch.cuda.is_bf16_supported(including_emulation=False)); "
   "print('transformers', transformers.__version__, 'peft', peft.__version__, 'accelerate', accelerate.__version__, "
   "'bitsandbytes', bitsandbytes.__version__, 'sacrebleu', sacrebleu.__version__, 'pandas', pandas.__version__)\"",
   log="logs/colab_environment.log")

## 3. Model weights (Hugging Face cache on the local disk)

1.3B: about 2.7 GB. 6.7B (only if `RUN_6_7B_4BIT` and not done yet): about 13.5 GB.

In [ ]:
models = [f"{M_13}@{REV_13}"]
if RUN_6_7B_4BIT and not has_metrics("zs_6.7b_small_4bit"):
    models.append(f"{M_67}@{REV_67}")
assert sh("python scripts/download_models.py --models " + " ".join(models)) == 0, "download failed"

## 4. Tests

In [ ]:
if not RUN_TESTS:
    skip("tests", "RUN_TESTS = False")
elif stage_done("tests"):
    skip("tests", "passed earlier")
else:
    started = time.time()
    finish("tests", sh("python -m pytest tests -q -p no:cacheprovider", log="logs/colab_tests.log"), started)

## 5. Preflight with the timing run

On the T4 the timing run uses the 1.3B model (the 6.7B model in 16 bits does not fit for training);
it prints examples/second and the largest batch that fits. The server window needs its own timing run
on the 4090s.

In [ ]:
if not RUN_PREFLIGHT:
    skip("preflight", "RUN_PREFLIGHT = False")
elif stage_done("preflight"):
    skip("preflight", "done earlier, see logs/preflight_colab.log")
else:
    started = time.time()
    model = f"{M_13}@{REV_13}"
    code_ = sh(f"python scripts/preflight.py --gpus 0 --data-dir data/processed/small --check-model {model} "
               f"--models {model} --timing-model {model} --timing-steps 30 --min-free-gb 12 --min-disk-gb 30",
               log="logs/preflight_colab.log")
    finish("preflight", code_, started)

## 6. Smoke rehearsal of the server script (WP5)

The identical plan with the 1.3B model everywhere, 64 pairs, 2-minute budgets, on GPU 0. Outputs go to
`runs_smoke/`, `logs_smoke/`, `results_smoke/` (never mixed with real runs). Re-running skips smoke runs
that already have `metrics.json`.

In [ ]:
if not RUN_SMOKE:
    skip("smoke", "RUN_SMOKE = False")
elif stage_done("smoke"):
    skip("smoke", "passed earlier")
else:
    started = time.time()
    code_ = sh(f"SMOKE=1 GPUS=0 BATCH={TRAIN_BATCH} ACCUM={TRAIN_ACCUM} EVAL_BATCH={EVAL_BATCH} "
               "PYTHON=python bash scripts/run_server.sh", log="logs/colab_smoke.log")
    finish("smoke", code_, started)

## 7. Insurance runs on the small tier (1.3B)

`zs_1.3b_small`: zero-shot on 500 val + 500 test pairs. `lora_1.3b_r16_small`: LoRA r16 on 4000 train pairs,
2 epochs, then evaluation on val and test. Each is skipped if its `metrics.json` already exists.

In [ ]:
COMMON = f"--model {M_13} --revision {REV_13} --data-dir data/processed/small"
if not RUN_INSURANCE:
    skip("zs_1.3b_small", "RUN_INSURANCE = False")
elif has_metrics("zs_1.3b_small"):
    skip("zs_1.3b_small", "metrics.json exists")
else:
    started = time.time()
    code_ = sh(f"python src/eval_repair.py {COMMON} --shots 0 --output runs/zs_1.3b_small "
               f"--batch-size {EVAL_BATCH}", log="logs/zs_1.3b_small.log")
    finish("zs_1.3b_small", code_, started)

In [ ]:
if not RUN_INSURANCE:
    skip("lora_1.3b_r16_small", "RUN_INSURANCE = False")
elif has_metrics("lora_1.3b_r16_small"):
    skip("lora_1.3b_r16_small", "metrics.json exists")
else:
    started = time.time()
    code_ = sh(f"python src/train_repair.py {COMMON} --output runs/lora_1.3b_r16_small --rank 16 "
               f"--epochs 2 --batch-size {TRAIN_BATCH} --accumulation-steps {TRAIN_ACCUM} "
               f"--eval-batch-size {EVAL_BATCH} --time-budget-min {TIME_BUDGET_MIN}",
               log="logs/lora_1.3b_r16_small.log")
    finish("lora_1.3b_r16_small", code_, started)

## 8. If time remains: 6.7B zero-shot in 4 bits on the small tier

A different run name (`zs_6.7b_small_4bit`) because the server's `zs_6.7b` is 16-bit on the full tier.

In [ ]:
if not RUN_6_7B_4BIT:
    skip("zs_6.7b_small_4bit", "RUN_6_7B_4BIT = False")
elif has_metrics("zs_6.7b_small_4bit"):
    skip("zs_6.7b_small_4bit", "metrics.json exists")
else:
    started = time.time()
    code_ = sh(f"python src/eval_repair.py --model {M_67} --revision {REV_67} --data-dir data/processed/small "
               f"--quantized --shots 0 --output runs/zs_6.7b_small_4bit --batch-size {EVAL_BATCH_6_7B}",
               log="logs/zs_6.7b_small_4bit.log")
    finish("zs_6.7b_small_4bit", code_, started)

## 9. Report and summary

`make_report.py` reads `runs/*/metrics.json`, picks the final configuration on **val** only, writes
`results/selection.json`, then the other tables. Download `MyDrive/CodeGuard/out/` afterwards.

In [ ]:
started = time.time()
finish("make_report", sh("python src/make_report.py --runs runs --results results", log="logs/colab_report.log"),
       started)
print()
for stage, status in STAGES:
    print(f"{stage:22s} {status}")
print()
for run in ("zs_1.3b_small", "lora_1.3b_r16_small", "zs_6.7b_small_4bit"):
    print(f"{run:22s} metrics.json: {'yes' if has_metrics(run) else 'NO'}")
print("\nEverything is in", OUT)